# Cross-validation and model comparison

This notebook compares logistic regression, random forest, and histogram
gradient boosting using stratified five-fold cross-validation.

Cross-validation is used to determine whether model performance remains
stable across different subsets of the training data.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))

In [2]:
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import ConfusionMatrixDisplay
from sklearn.model_selection import StratifiedKFold, cross_validate

from predictive_maintenance.data import load_data
from predictive_maintenance.evaluate import calculate_metrics
from predictive_maintenance.features import create_features
from predictive_maintenance.train import (
    create_gradient_boosting_model,
    create_logistic_model,
    create_random_forest_model,
    split_data,
)

In [3]:
df = load_data()
X, y = create_features(df)

X_train, X_test, y_train, y_test = split_data(X, y)

print(f"Training observations: {len(X_train)}")
print(f"Test observations: {len(X_test)}")
print(f"Training failures: {y_train.sum()}")
print(f"Test failures: {y_test.sum()}")

Loading local dataset from /Users/alexanderkorolev/Desktop/predictive-maintenance-ai4i/data/raw/ai4i2020.csv
Training observations: 8000
Test observations: 2000
Training failures: 271
Test failures: 68


In [4]:
models = {
    "Logistic regression": create_logistic_model(X_train),
    "Random forest": create_random_forest_model(X_train),
    "Gradient boosting": create_gradient_boosting_model(X_train),
}

In [5]:
cross_validation = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "precision": "precision",
    "recall": "recall",
    "f1_score": "f1",
    "pr_auc": "average_precision",
}

In [ ]:
rows = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cross_validation,
        scoring=scoring,
        n_jobs=-1,
    )

    rows.append(
        {
            "model": model_name,
            "precision_mean": scores["test_precision"].mean(),
            "precision_std": scores["test_precision"].std(),
            "recall_mean": scores["test_recall"].mean(),
            "recall_std": scores["test_recall"].std(),
            "f1_mean": scores["test_f1_score"].mean(),
            "f1_std": scores["test_f1_score"].std(),
            "pr_auc_mean": scores["test_pr_auc"].mean(),
            "pr_auc_std": scores["test_pr_auc"].std(),
        }
    )

cv_results = pd.DataFrame(rows)
cv_results.set_index("model").round(3)

In [ ]:
plot_results = (
    cv_results.set_index("model")[
        [
            "precision_mean",
            "recall_mean",
            "f1_mean",
            "pr_auc_mean",
        ]
    ]
    .rename(
        columns={
            "precision_mean": "Precision",
            "recall_mean": "Recall",
            "f1_mean": "F1-score",
            "pr_auc_mean": "PR-AUC",
        }
    )
)

plot_results.plot(
    kind="bar",
    figsize=(11, 6),
)

plt.title("Five-Fold Cross-Validation Results")
plt.ylabel("Mean score")
plt.xlabel("")
plt.ylim(0, 1)
plt.xticks(rotation=0)
plt.legend(loc="upper right")
plt.tight_layout()
plt.show()

In [ ]:
best_model_name = cv_results.loc[
    cv_results["pr_auc_mean"].idxmax(),
    "model",
]

print(f"Best model by mean PR-AUC: {best_model_name}")

In [ ]:
best_model = models[best_model_name]

best_model.fit(
    X_train,
    y_train,
)

In [ ]:
test_metrics = calculate_metrics(
    best_model,
    X_test,
    y_test,
)

pd.Series(test_metrics).round(3)

In [ ]:
ConfusionMatrixDisplay.from_estimator(
    best_model,
    X_test,
    y_test,
    display_labels=["No failure", "Failure"],
    cmap="Purples",
)

plt.title(f"{best_model_name} – Test Confusion Matrix")
plt.tight_layout()
plt.show()